# MATS 12 — value-leakage mechanism on Qwen3.5 (Colab)

Runs the `~/mats12` pipeline on a Colab GPU. **Runtime → Change runtime type → GPU** (a free T4 works in fp16; an L4/A100 is faster and uses bf16). Everything up to the section marked *clock starts* is uncounted setup under Nanda's rules.

Repo: https://github.com/martinherje/mats12 (private). Primer: vault `plans/applications/MATS 12 - Value Leakage Primer.md`. Design sheet: `journal/design-questions-value-leakage.md` — answer it before the go/no-go.

**Persistence:** Colab VMs vanish. Cell 2 mounts Google Drive and keeps `data/raw`, `data/processed`, `figures` and `journal` there, so runs survive and the journal is one file across sessions.

In [ ]:
#@title 1 · GPU check
import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "no GPU — Runtime → Change runtime type → GPU")
print("cuda:", torch.cuda.is_available(), "| capability:", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None, "(>=8 → bf16, else fp16)")

In [ ]:
#@title 2 · Drive + repo (private repo: put a GitHub token in Colab Secrets 🔑 as GITHUB_TOKEN — fine-grained, Contents: read/write on martinherje/mats12)
import os, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/mats12_runs'   #@param {type:"string"}
os.makedirs(DRIVE_DIR, exist_ok=True)
try:
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = f"https://{tok}@github.com/martinherje/mats12.git" if tok else "https://github.com/martinherje/mats12.git"
if not os.path.exists('/content/mats12'):
    subprocess.run(["git", "clone", "-q", url, "/content/mats12"], check=True)
else:
    subprocess.run(["git", "-C", "/content/mats12", "pull", "-q"], check=True)
%cd /content/mats12
!git log --oneline -1
!bash scripts/colab_setup.sh "$DRIVE_DIR" 

In [ ]:
#@title 3 · Smoke test — loads the model, prints layers/width/peak memory (uncounted)
MODEL = "Qwen/Qwen3.5-4B"   #@param ["Qwen/Qwen3.5-4B", "Qwen/Qwen3.5-9B"]
!python scripts/gpu_smoke.py --model $MODEL

## ⏱ Clock starts here
Open `journal/design-questions-value-leakage.md` (Files panel on the left → `mats12/journal`; it lives on Drive now), answer the thirteen questions in a sentence each, copy the answers into `journal/highlights.md` with the date. Start Toggl. Then run the go/no-go.

In [ ]:
#@title 4 · Go/no-go, thinking OFF — concrete, then abstract and equal reusing the thresholds (~15–30 min on a T4)
N = 20         #@param {type:"integer"}
RUN = "gonogo_4b"   #@param {type:"string"}
!python scripts/donation_bet.py --backend local --model $MODEL --run $RUN --variant concrete_amf_kw --think off --n-baseline $N --n-per-direction $N --batch-size 8
!python scripts/donation_bet.py --backend local --model $MODEL --run {RUN}_abstract --variant abstract --think off --n-per-direction $N --reuse-thresholds $RUN --batch-size 8
!python scripts/donation_bet.py --backend local --model $MODEL --run {RUN}_equal --variant equal_dwb_imc --think off --n-per-direction $N --reuse-thresholds $RUN --batch-size 8

In [ ]:
#@title 5 · Headline table (0 = no leak; expect abstract ≥ concrete > equal ≈ 0)
import json, pandas as pd
rows = []
for r, v in [(RUN, "concrete_amf_kw"), (f"{RUN}_abstract", "abstract"), (f"{RUN}_equal", "equal_dwb_imc")]:
    try:
        for think, s in json.load(open(f"data/processed/donation_bet_{r}.json"))["headline"].items():
            rows.append({"variant": v, "think": think, "bias": s["bias"], "ci_lo": s["ci_lo"], "ci_hi": s["ci_hi"], "n": s["n"], "unparsed": s["unparsed_frac"]})
    except FileNotFoundError:
        rows.append({"variant": v, "think": "-", "bias": None})
display(pd.DataFrame(rows).round(3))

In [ ]:
#@title 6 · Hand-check: 20 random raw answers per condition with the parsed number — READ THESE, then log the count in journal/verification-log.md
import json, random
random.seed(0)
rows = [json.loads(l) for l in open(f"data/raw/donation_bet_{RUN}.jsonl")]
for cond in ("baseline", "above_good", "below_good"):
    sub = [r for r in rows if r["condition"] == cond]
    print(f"\n===== {cond}  (n={len(sub)}) =====")
    for r in random.sample(sub, min(20, len(sub))):
        print(f"[{r['question']:10s}] est={r['estimate']!s:>14} ({r['parse_method']:15s}) | ...{r['answer'][-160:].replace(chr(10),' ')}")

In [ ]:
#@title 7 · (optional, slow) Go/no-go with thinking ON, smaller N — is the leak reasoning-mediated?
N_ON = 10   #@param {type:"integer"}
!python scripts/donation_bet.py --backend local --model $MODEL --run {RUN}_thinkon --variant concrete_amf_kw --think on --n-per-direction $N_ON --reuse-thresholds $RUN --max-new-tokens 1500 --batch-size 8
import json; print(json.load(open(f"data/processed/donation_bet_{RUN}_thinkon.json"))["headline"])

In [ ]:
#@title 8 · Activations at the answer-start position → per-layer difference-of-means direction (favoured side) + topic direction (bet vs no bet)
!python scripts/extract_activations.py --model $MODEL --scenarios data/scenarios_{RUN}.csv --template chat --generation-prompt --enable-thinking off --run $RUN
!python scripts/make_direction.py --run $RUN --label good_side --filter "bet==1" --out direction_{RUN}_good_side
!python scripts/make_direction.py --run $RUN --label bet --out direction_{RUN}_bet

In [ ]:
#@title 9 · The causal move: ablate the favoured-side direction at a layer band; then the random-direction control; then the topic (bet) direction
LAYERS = "16,20,24"   #@param {type:"string"}
!python scripts/donation_bet.py --backend local --model $MODEL --run {RUN}_abl --variant concrete_amf_kw --think off --n-per-direction $N --reuse-thresholds $RUN --ablate data/processed/direction_{RUN}_good_side.npz --layers $LAYERS --mode ablate --batch-size 8
!python scripts/donation_bet.py --backend local --model $MODEL --run {RUN}_rand --variant concrete_amf_kw --think off --n-per-direction $N --reuse-thresholds $RUN --layers $LAYERS --mode random --batch-size 8
!python scripts/donation_bet.py --backend local --model $MODEL --run {RUN}_topic --variant concrete_amf_kw --think off --n-per-direction $N --reuse-thresholds $RUN --ablate data/processed/direction_{RUN}_bet.npz --layers $LAYERS --mode ablate --batch-size 8
import json, pandas as pd
display(pd.DataFrame([{"run": r, **json.load(open(f"data/processed/donation_bet_{r}.json"))["headline"]["off"]} for r in [RUN, f"{RUN}_abl", f"{RUN}_rand", f"{RUN}_topic"]]).round(3))

In [ ]:
#@title 10 · Fact-retention check: with the direction ablated, does the model still say which charity does more good?
import sys, json, numpy as np, torch; sys.path.insert(0, "scripts")
from common import load_model, pick_device, pick_dtype
from steer import Intervention
Q = json.load(open("data/donation_bet_questions.json")); v = Q["variants"]["concrete_amf_kw"]
prompt = Q["fact_check_prompt"].format(favoured=v["favoured"], other=v["other"])
dev = pick_device("auto"); tok, model = load_model(MODEL, dev, pick_dtype("auto", dev)); tok.padding_side = "left"
dirs = np.load(f"data/processed/direction_{RUN}_good_side.npz")["dirs"]
def ask(n=10, ablate=False):
    text = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    enc = tok([text] * n, return_tensors="pt", padding=True).to(dev)
    ctx = [Intervention(model, [int(l)], torch.tensor(dirs[int(l)]), mode="ablate") for l in LAYERS.split(",")] if ablate else []
    for c in ctx: c.__enter__()
    try:
        with torch.no_grad(): out = model.generate(**enc, max_new_tokens=20, do_sample=True, temperature=1.0, pad_token_id=tok.pad_token_id)
    finally:
        for c in ctx: c.__exit__(None, None, None)
    return [tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]
print("prompt:", prompt)
print("no ablation :", ask())
print("ablated     :", ask(ablate=True))

In [ ]:
#@title 11 · Push the journal back to GitHub (needs GITHUB_TOKEN in Secrets). Outputs already live on Drive.
# journal/ is a symlink to Drive inside the repo; git must see a real directory, so copy it in, commit, then restore the link.
!git config user.email "mherje@live.com" && git config user.name "Martin Herje"
!rm -f journal && cp -r "$DRIVE_DIR/journal" journal && git add journal && (git commit -qm "journal: Colab session" || true) && git push -q origin main && echo pushed
!rm -rf journal && ln -s "$DRIVE_DIR/journal" journal